setup

In [7]:
import pandas as pd
import numpy as np
import xgboost as xgb
import sys
from pathlib import Path
import importlib
import explainability.translator
importlib.reload(explainability.translator)
from explainability.translator import get_shap_explanation, format_whatsapp_message

sys.path.insert(0, str(Path.cwd().parent))

from feature_store.engineer import build_features, get_feature_columns
from feature_store.sources.weather import generate_simulated_weather
from model.evaluate import compute_wmape
from explainability.translator import get_shap_explanation, format_whatsapp_message

DATA_RAW = Path(r"C:\Users\syeds\OneDrive\Desktop\nboracle\nb_oracle\data\raw\store-sales-time-series-forecasting")

train = pd.read_csv(DATA_RAW / "train.csv", parse_dates=["date"])
holidays = pd.read_csv(DATA_RAW / "holidays_events.csv", parse_dates=["date"])
weather = generate_simulated_weather(train["date"].sort_values().unique())

store44 = train[train.store_nbr == 44].copy()
bev = (store44[store44.family == "BEVERAGES"]
       .sort_values("date").reset_index(drop=True))

features = build_features(bev, holidays, weather_df=weather).dropna()
feat_cols = get_feature_columns(features)

SPLIT_DATE = "2017-07-15"
train_df = features[features.index < SPLIT_DATE]
test_df = features[features.index >= SPLIT_DATE]

X_train, y_train = train_df[feat_cols], train_df["sales"]
X_test, y_test = test_df[feat_cols], test_df["sales"]

model = xgb.XGBRegressor(
    objective="reg:tweedie", tweedie_variance_power=1.6,
    n_estimators=500, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.7, min_child_weight=10,
    reg_alpha=0.1, reg_lambda=1.0, random_state=42,
    early_stopping_rounds=30)

model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
predictions = np.maximum(model.predict(X_test), 0)
print("✅ Model trained. Ready to test explanations.")

✅ Model trained. Ready to test explanations.


testing new explanations on interesting days

In [8]:
# Test on 7 different days — mix of high, low, and normal sales
test_days = [
    y_test.values.argmax(),     # Highest sales
    y_test.values.argmin(),     # Lowest sales
    3, 8, 15, 22, 28            # Various days
]

print("=" * 65)
print("  IMPROVED EXPLANATIONS — Store 44, Beverages")
print("=" * 65)

for idx in test_days:
    date = y_test.index[idx]
    features_row = X_test.iloc[idx]
    actual = y_test.values[idx]
    pred = predictions[idx]
    
    result = get_shap_explanation(
        model, features_row, feat_cols, date, pred, actual)
    
    print(f"\n📅 {date.strftime('%A, %B %d, %Y')}")
    print(f"   Predicted: {pred:,.0f}  |  Actual: {actual:,.0f}")
    print(f"   Confidence: {result['confidence'].upper()}")
    print(f"\n   💬 Summary: {result['summary']}")
    print(f"   📝 {result['explanation']}")
    print("-" * 65)

  IMPROVED EXPLANATIONS — Store 44, Beverages

📅 Sunday, July 30, 2017
   Predicted: 13,587  |  Actual: 18,340
   Confidence: HIGH

   💬 Summary: Looking like a strong Sunday — forecast is ~13,587 units.
   📝 Sundays are one of your busiest days — expect higher than average traffic. Your sales have been running strong this week (averaging 10,292/day), which is pushing the forecast up.
-----------------------------------------------------------------

📅 Tuesday, July 25, 2017
   Predicted: 8,160  |  Actual: 6,862
   Confidence: MODERATE

   💬 Summary: Shaping up to be a typical Tuesday — forecast is ~8,160 units.
   📝 Tuesdays tend to be on the quieter side for this category. Your sales have been running strong this week (averaging 10,594/day), which is pushing the forecast up. Also worth noting: this category is running a promotion.
-----------------------------------------------------------------

📅 Tuesday, July 18, 2017
   Predicted: 8,452  |  Actual: 8,366
   Confidence: MODERATE



whatsapp message testing

In [10]:
# See what the actual WhatsApp messages would look like

print("=" * 65)
print("  SAMPLE WHATSAPP MESSAGES")
print("=" * 65)

for idx in [y_test.values.argmax(), 15, y_test.values.argmin()]:
    date = y_test.index[idx]
    features_row = X_test.iloc[idx]
    actual = y_test.values[idx]
    pred = predictions[idx]
    
    result = get_shap_explanation(
        model, features_row, feat_cols, date, pred, actual)
    
    # Simulate inventory (random for demo)
    fake_inventory = int(pred * np.random.uniform(0.7, 1.3))
    
    msg = format_whatsapp_message(
        store_name="Store 44",
        category="Beverages",
        date=date,
        prediction=pred,
        explanation_result=result,
        current_inventory=fake_inventory
    )
    
    print(f"\n{msg}")
    print("-" * 65)

  SAMPLE WHATSAPP MESSAGES

🔮 *Store 44 — Beverages*
📅 Sunday, July 30

*Looking like a strong Sunday — forecast is ~13,587 units.*

Sundays are one of your busiest days — expect higher than average traffic. Your sales have been running strong this week (averaging 10,292/day), which is pushing the forecast up.

🟢 Confidence: High

⚠️ *Heads up:* You have 12,173 units on hand but we're forecasting 13,587. Consider ordering 1,414 more.
-----------------------------------------------------------------

🔮 *Store 44 — Beverages*
📅 Sunday, July 30

*Looking like a strong Sunday — forecast is ~13,587 units.*

Sundays are one of your busiest days — expect higher than average traffic. Your sales have been running strong this week (averaging 10,292/day), which is pushing the forecast up.

🟢 Confidence: High

📦 Stock is tight — you have 14,592 units with only 1,005 units of buffer.
-----------------------------------------------------------------

🔮 *Store 44 — Beverages*
📅 Tuesday, July 25

*Sha